In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads credentials from a local .env file (see .env.example)
import tweepy as tw
import pandas as pd
import plotly
from plotly.offline import plot
import plotly.graph_objects as go
from twarc import Twarc2
import json
import csv

from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

import pandas as pd
import numpy as np
import re

from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import string
import nltk
nltk.download('words')
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
all_stopwords = stopwords.words('english')
all_stopwords.remove('not')

import seaborn as sns
from wordcloud import WordCloud
import matplotlib.pyplot as plt

from IPython.display import display, clear_output

from ipywidgets import widgets
sleep_on_rate_limit=False

from textblob import TextBlob

import plotly.express as px
import cufflinks as cf
from plotly.offline import download_plotlyjs, plot,iplot
import plotly.subplots as tls

import datetime
from datetime import datetime, timedelta

import pickle

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
vds=SentimentIntensityAnalyzer()

import spacy

from fuzzywuzzy import fuzz

from tabulate import tabulate

#import func

#import warnings
#warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

In [ ]:
# a function to remove non-english words, stopwords, handlers, punctuations and converting words with similar meanings to their common root form 
words = set(nltk.corpus.words.words())
all_stopwords = stopwords.words('english')
all_stopwords.remove('not')
all_stopwords.remove('no')
stop = set(all_stopwords)
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()

def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    handler1_free = " ".join([i for i in stop_free.lower().split() if '@' not in i])
    handler2_free = " ".join([i for i in handler1_free.lower().split() if 'http' not in i])
    handler3_free = " ".join([i for i in handler2_free.lower().split() if '#' not in i])
    handler4_free = " ".join([i for i in handler3_free.lower().split() if '/' not in i])
    punc_free = ''.join(ch for ch in handler4_free if ch not in exclude)
    wn_lemmatized1 = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    wn_lemmatized2 = " ".join(lemma.lemmatize(word, 'v') for word in wn_lemmatized1.split())
    wn_lemmatized3 = " ".join(lemma.lemmatize(word, 'a') for word in wn_lemmatized2.split())
    normalized = " ".join([i for i in wn_lemmatized3.lower().split() if i in words])
    return normalized

# a function to remove all types of emojis
def deEmojify(text):
    regrex_pattern = re.compile(pattern = "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002500-\U00002BEF"  # chinese char
        u"\U00002702-\U000027B0"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642" 
        u"\u2600-\u2B55"
        u"\u200d"
        u"\u23cf"
        u"\u23e9"
        u"\u231a"
        u"\ufe0f"  # dingbats
        u"\u3030"
                           "]+", flags = re.UNICODE)
    return regrex_pattern.sub(r'',text)


# a function to remove non-english words, stopwords, handlers, punctuations and converting words with similar meanings to their common root form 
lemma = WordNetLemmatizer()
def clean_ent(doc):
    handler1_free = " ".join([i for i in doc.split() if '@' not in i])
    handler2_free = " ".join([i for i in handler1_free.split() if 'http' not in i])
    handler3_free = " ".join([i for i in handler2_free.split() if '#' not in i])
    handler4_free = " ".join([i for i in handler3_free.split() if i[0] not in '/'])
    noneng_free = re.sub('[^a-zA-Z ?!.-_]', ' ', handler4_free)
    return noneng_free

In [ ]:
def contains_word(s, w):
    return (' ' + w + ' ') in (' ' + s + ' ')

def preprocess_review(review):
    review = review.lower()
    review = review.replace("-",'')
    review = review.replace(".",'')
    review = review.replace(",",'')
    return review

class my_dictionary(dict):
    def __init__(self):
        self = dict()
    def add(self, key, value):
        self[key] = value
        
#this function takes one parameter entities_df is the dataframe of predefined entities having one column named "entity" and return the dictionary  
def make_dictionary(entities_df):
    dictionary=my_dictionary()
    for index, row in entities_df.iterrows(): 
        key=row["entity"]
        i=0
        dictionary.add(key,i)
    return dictionary

#this function takes df as dataframe parameter that contains review having one column named "Tweets"
# and the other parameter entities_df is the dataframe of predefined entities having one column named "entity"
def get_predefined_entities_count(df,entities_df):
    entities=make_dictionary(entities_df)
    for index, row in df.iterrows(): 
        review=preprocess_review(row["Tweets"])
        for entity, count in entities.items():
            if(contains_word(review,entity)):
                count=count+1
                naya={entity:count}
                entities.update(naya)
    return entities


def get_predefined_entities_count_sentiment_wise_OOP(df,entities_df):
    positive_df=df.loc[df['Sentiment'] == "Positive"]
    negative_df=df.loc[df['Sentiment'] == "Negative"]
    neutral_df = df.loc[df['Sentiment'] == "Neutral"]
    positive_entities=get_predefined_entities_count(positive_df,entities_df)
    negative_entities=get_predefined_entities_count(negative_df,entities_df)
    neutral_entities=get_predefined_entities_count(neutral_df,entities_df)
    return positive_entities,negative_entities,neutral_entities

In [ ]:
def final_predefined_entities_count(df, entities_df):
    positive_dict_OOP, negative_dict_OOP, neutral_dict_OOP = get_predefined_entities_count_sentiment_wise_OOP(df, entities_df)
    pos=[]
    neg=[]
    neu=[]
    for index, row in entities_df.iterrows():
        entity=row['entity']
        pos.append(positive_dict_OOP[entity])
        neg.append(negative_dict_OOP[entity])
        neu.append(neutral_dict_OOP[entity])
    entities_df['Positive']=pos
    entities_df['Negative']=neg
    entities_df['Neutral']=neu
    final_entities_count = entities_df.groupby(['Name']).agg({'Positive':pd.Series.sum,'Negative':pd.Series.sum,'Neutral':pd.Series.sum})
    return final_entities_count

In [ ]:
# function to collect tweets and associated information using tweepy api
def collect_tweets(consumer_key, consumer_secret, access_token, access_token_secret, search_words):
    auth = tw.OAuthHandler(consumer_key, consumer_secret)
    auth.set_access_token(access_token, access_token_secret)
    api = tw.API(auth, wait_on_rate_limit=True)
    auth = tw.API(auth, proxy='https://your_proxy.server:port')    
    tweets = tw.Cursor(api.search_tweets,
                      q=search_words,
                      lang="en",
                      ).items(250)
    return tweets

# function to print top 10 tweets from collected tweets
def print_ten_tweets(tweets):
    print("Most recent tweets: \n")    
    i = 1
    # Iterate and print tweets
    for tweet in tweets:
        print(i,end='  ')
        i+=1
        print(tweet.text)
        if(i==11):
            break 

# function to store all the tweets related data in a dataframe
def tweets_dataframe(tweets):
    twt=[]
    creation = []
    retweet_count=[]
    likes_count=[]
    for tweet in tweets:
        twt.append(tweet.text)
        creation.append(tweet.created_at.strftime("%Y-%m-%d")) 
        retweet_count.append(tweet.retweet_count)
        likes_count.append(tweet.favorite_count)
    df_tweepy = pd.DataFrame(data={'Date':creation, 'Tweets':twt, 'Retweet_count':retweet_count, 'Likes_count':likes_count})
    return df_tweepy

In [ ]:
# function to preprocess tweets
def preprocess(df):
    df['Tweets_processed'] = [clean(str(df['Tweets'][i])) for i in range(len(df))]
    df['Tweets_processed'] = [deEmojify(str(df['Tweets_processed'][i])) for i in range(len(df))]    
    return df

# function to preprocess tweets for entity extraction
def preprocess_for_entity(df):
    df.drop(columns=['Tweets_processed'], inplace=True)
    df['Tweets_processed'] = [clean_ent(str(df['Tweets'][i])) for i in range(len(df))]
    df['Tweets_processed'] = [deEmojify(str(df['Tweets_processed'][i])) for i in range(len(df))]    
    return df

# function to compute polarity and subjectivity scores
def ps_scores(df):
    df['Polarity'] = [TextBlob(str(df['Tweets_processed'][i])).sentiment.polarity for i in range(len(df))]
    df['Subjectivity'] = [TextBlob(str(df['Tweets_processed'][i])).sentiment.subjectivity for i in range(len(df))]
    print('\n')
    print('\n')
    print("Average Polarity: ", round(float(df['Polarity'].mean()),2), '\n')
    print("Average Subjectivity: ", round(float(df['Subjectivity'].mean()),2), '\n')   
    print('\n')
    df['Polarity_Quantiles'] = pd.cut(df['Polarity'], bins=[-1, -0.5, 0.0, 0.5, 1.1], right=False)
    #df['Polarity_Quantiles'] = pd.qcut(df['Polarity'], q=6, bins=[-1, -0.5, 0.0, 0.5, 1.0], precision=0, duplicates='drop')
    #df['Polarity_Quantiles'] = np.round(df['Polarity_Quantiles'], 2)
    df.Polarity_Quantiles.value_counts().sort_index(ascending=True).plot(kind='bar', title='Sentiment quantiles count');
    #s.index = s.index.map(lambda l : np.around(l,2))
    plt.xticks(rotation=0, ha='left')
    plt.show()
    print("\n")
    df_perc = df[['Polarity']].dropna().quantile([0.0,0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.90,0.95,1.0])
    df_perc.rename(index={0.0:'0%', 0.01:'1%', 0.05:'5%', 0.1:'10%', 0.2:'20%', 0.3:'30%', 0.4:'40%', 0.5:'50%', 0.6:'60%', 0.7:'70%', 0.8:'80%', 0.9:'90%', 0.95:'95%', 1:'100%'}, inplace=True)
    df_perc.index.names = ['Percentile']
    df_perc['Polarity'] = np.round(df_perc['Polarity'], 3)
    print(tabulate(df_perc, headers='keys', tablefmt='fancy_grid'))
    print("NOTE: Polarity here means sentiment score of a tweet which ranges from -1 (very negative sentiment) to +1 (very positive sentiment).")
    print("\n\n")
    #fig = tls.make_subplots(rows=1, cols=2, subplot_titles=('Sentiment quantiles count', 'Sentiment percentiles'))
    #fig.add_trace(fig1, 1, 1)
    #fig.add_trace(fig2, 1, 2)
    #fig.update_layout(showlegend=False)
    #iplot(fig)
    return df

# function to predict sentiment categories of collected tweets using trained Vader model
def predict_sentiment_cat(df):
    # features creation using vader scores
    compound=[]
    neg=[]
    neu=[]
    pos=[]
    for ind in df.index:
        review = df['Tweets_processed'][ind]          
        c=vds.polarity_scores(review)
        compound.append(c['compound'])
        neg.append(c['neg'])
        neu.append(c['neu'])
        pos.append(c['pos'])

    X = pd.DataFrame(
    {'compound': compound,
     'negative': neg,
     'neutral': neu,
     'positive':pos,
    })

    # load the model from disk
    loaded_model = pickle.load(open('finalized_model.sav', 'rb'))

    # predicting sentiment category (0, 1 or 2) using trained vader model
    prediction_test = loaded_model.predict(X)
    df['Sentiment'] = prediction_test

    for i in range(len(df)):
        if df['Sentiment'][i]==1:
            df['Sentiment'][i] = 'Neutral'
        elif df['Sentiment'][i]==0:
            df['Sentiment'][i] = 'Negative'
        elif df['Sentiment'][i]==2:
            df['Sentiment'][i] = 'Positive'
        else:
            break
    return df

In [ ]:
def daily_hourly_trend_last7days(client, search_words_wo_filter):
    query = search_words_wo_filter
    search_results = client.counts_recent(query=query,granularity='hour')

    for page in search_results:
        data = page['data']
        break

    day = []
    tweet_counts = []

    for d in data:
        day.append(d['start'][:10])
        tweet_counts.append(d['tweet_count'])

    df = pd.DataFrame({'day':day, 'tweet_counts':tweet_counts})
    df_count = df.groupby(by=['day']).agg({'tweet_counts':'sum'})
    # Plot of daily count of Tweets
    fig1 = px.line(df_count['tweet_counts'])
    
    print("Total Number of Tweets: ", df_count['tweet_counts'].sum(), '\n')             

    # Plot of hourly count of tweets
    fig2 = go.Figure(data=[go.Bar(x= day, y = tweet_counts)])
    
    # to show two interactive plots together
    fig = tls.make_subplots(rows=1, cols=2, subplot_titles=('Tweet trend over last 7 days', 'Hourly trend over last 7 days'))        
    fig.add_trace(fig1['data'][0], 1, 1)
    fig.add_trace(fig2['data'][0], 1, 2)
    fig.update_layout(showlegend=False)
    iplot(fig)  

def append_tweets_to_consolidated_dataset(dataset, df):        
    # adding 250 tweets to consolidated dataset
    for i in range(len(df)):
        dataset = dataset.append({'Date': df['Date'][i], 'Passenger Feedback': df['Tweets'][i], 'Sentiment': df['Sentiment'][i]}, ignore_index=True)

    for k in range(len(dataset)):
        dataset.loc[k, 'Date'] = pd.to_datetime(dataset.loc[k, 'Date']).strftime('%Y-%m-%d')

    # Daily count of tweets based on sentiments
    df_trend = dataset.groupby(by=['Date', 'Sentiment']).agg({'Passenger Feedback':'count'})
    df_trend = df_trend.reset_index()
    df_trend = df_trend[df_trend['Date'] >= '2021-10-01']

    return df_trend

def stacked_chart(df):
    # Creating 3 columns: 'Negative', 'Neutral', 'Positive' - containing daily tweets counts
    df1 = pd.DataFrame(df[df['Sentiment']=='Negative'].drop(['Sentiment'], axis=1).set_index('Date')['Passenger Feedback'])
    df1.rename(columns={'Passenger Feedback': 'Negative'}, inplace=True)
    df2 = pd.DataFrame(df[df['Sentiment']=='Neutral'].drop(['Sentiment'], axis=1).set_index('Date')['Passenger Feedback'])
    df2.rename(columns={'Passenger Feedback': 'Neutral'}, inplace=True)
    df3 = pd.DataFrame(df[df['Sentiment']=='Positive'].drop(['Sentiment'], axis=1).set_index('Date')['Passenger Feedback'])
    df3.rename(columns={'Passenger Feedback': 'Positive'}, inplace=True)

    df_merge1 = pd.merge(df1, df2, left_index=True, right_index=True, how='outer')
    df_merge2 = pd.merge(df_merge1, df3, left_index=True, right_index=True, how='outer')
    df_merge2 = df_merge2.fillna(0)
    df_stacked = df_merge2.iloc[-7:,:]
    # plotting horizontal stacked chart
    df_stacked.plot.barh(stacked=True, figsize=(20,6))
    plt.title('Number of Tweets')
    plt.ylabel('')
    plt.show()

In [ ]:
def plot_wordclouds(df):
    data_plot= df['Tweets_processed'].astype(str)
    # general wordcloud
    wc = WordCloud(max_words = 100 , width = 1600 , height = 800,
                   collocations=False).generate(" ".join(data_plot))
    
    # Wordcloud for Positive Tweets
    df_pos = df[df['Sentiment']=='Positive']
    data_plot_p = df_pos['Tweets_processed'].astype(str)
    wc_p = WordCloud(max_words = 100 , width = 1600 , height = 800,
                     collocations=False).generate(" ".join(data_plot_p))

    # Wordcloud for Negative Tweets
    if len(df[df['Sentiment']=='Negative']) > 0:
        df_neg = df[df['Sentiment']=='Negative']
        data_plot_n = df_neg['Tweets_processed'].astype(str)
        wc_n = WordCloud(max_words = 100 , width = 1600 , height = 800,
                         collocations=False).generate(" ".join(data_plot_n))

    # Wordcloud for Neutral Tweets
    df_neu = df[df['Sentiment']=='Neutral']
    data_plot_neu = df_neu['Tweets_processed'].astype(str)
    wc_neu = WordCloud(max_words = 100 , width = 1600 , height = 800,
                       collocations=False).generate(" ".join(data_plot_neu))
    
    # plot of Overall wordcloud
    fig_wc = px.imshow(wc).data[0]
    fig = tls.make_subplots(rows=1, cols=1)
    fig.add_trace(fig_wc, 1,1) 
    iplot(fig)

    # plots of negative/neutral/positive wordclouds
    if len(df[df['Sentiment']=='Negative']) > 0: 
        fig_sentiment = tls.make_subplots(rows=1, cols=3, subplot_titles=('Negative', 'Neutral', 'Positive'))
        fig_p = px.imshow(wc_p).data[0]
        fig_n = px.imshow(wc_n).data[0]
        fig_neu = px.imshow(wc_neu).data[0]
        fig_sentiment.add_trace(fig_n, 1,1)
        fig_sentiment.add_trace(fig_neu, 1,2)
        fig_sentiment.add_trace(fig_p, 1,3)
        iplot(fig_sentiment)
    else:
        fig_sentiment = tls.make_subplots(rows=1, cols=2, subplot_titles=('Positive', 'Neutral'))
        fig_p = px.imshow(wc_p).data[0]
        fig_neu = px.imshow(wc_neu).data[0]
        fig_sentiment.add_trace(fig_p, 1,1)
        fig_sentiment.add_trace(fig_neu, 1,2)
        iplot(fig_sentiment)
   
def sentiment_trend(df):
    # Plot of Tweet Sentiment Trend (for full conslidated dataset Oct 1 onwards)
    plt.figure(figsize=(10,5))
    df[df['Sentiment']=='Negative'].drop(['Sentiment'], axis=1).set_index('Date')['Passenger Feedback'].plot(label='Negative', color='blue')
    df[df['Sentiment']=='Positive'].drop(['Sentiment'], axis=1).set_index('Date')['Passenger Feedback'].plot(label='Positive', color='green')
    df[df['Sentiment']=='Neutral'].drop(['Sentiment'], axis=1).set_index('Date')['Passenger Feedback'].plot(label='Neutral', color='orange')
    plt.ylabel('Count')
    plt.legend(loc='best', ncol=3)
    plt.title('Tweet Sentiment Trend')
    plt.show()

In [ ]:
def dynamic_entity_table(df):    
    NER = spacy.load("en_core_web_sm", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"])
    df['Description'] = [NER(str(df['Tweets_processed'][i])) for i in range(len(df))]
    df['Entities'] = [df['Description'][i].ents for i in range(len(df))]

    for k in range(len(df)):
        df['Entities'][k] = list(df['Entities'][k])

    for j in range(len(df)):
        for i in range(len(df['Entities'][j])):
            df['Entities'][j][i] = str(df['Entities'][j][i])

    df['Entities_Count'] = np.nan

    for j in range(len(df)):
        word_count = dict()   
        for i in range(len(df['Entities'][j])):
            word_count[df['Entities'][j][i]] = df['Entities'][j].count(df['Entities'][j][i])
        df.loc[j, 'Entities_Count'] = [str(word_count)]

    df['Entity'] = np.nan

    for i in range(len(df)):
        if len(df['Entities'][i]) == 0:
            continue
        elif len(df['Entities'][i]) == 1:
            df['Entity'][i] = df['Entities'][i][0]
        else:
            df['Entity'][i] = df['Entities'][i][0]
            df = df.append([df.iloc[i]]*(len(df['Entities'][i])-1))
            df.reset_index(inplace=True, drop=True)
            for j in np.arange(1, len(df['Entities'][i]), 1):
                df.loc[len(df)-j, 'Entity'] = df['Entities'][i][j]

    df_ent_tr = df.groupby(by=['Entity', 'Sentiment']).Sentiment.count().unstack(level=1)
    print("\n")
    if 'Negative' not in df_ent_tr.columns:
        df_ent_tr['Negative'] = 0
        df_ent_tr.insert(0, 'Negative', df_ent_tr.pop('Negative'))

    df_ent_tr.fillna(0, inplace=True)
    df_ent_tr.sort_values(by=['Neutral', 'Positive', 'Negative'], ascending=False, inplace=True)
    df_ent_tr = df_ent_tr.astype('int')
    df_ent_tr.to_excel('df03.xlsx')
    df_all_ent = pd.read_excel('df03.xlsx')

    df_all_ent['Entity'] = df_all_ent['Entity'].str.lower().str.strip().str.replace(" ","")

    # remove entites containing numbers
    for i in range(len(df_all_ent['Entity'])):
        if any(str(j) in str(df_all_ent['Entity'][i]) for j in list(range(10))):
            df_all_ent.loc[i, 'Entity'] = np.nan
        else:
            continue

    df_all_ent.dropna(inplace=True)
    df_all_ent.reset_index(inplace=True, drop=True)

    # to find out indices of similar entities
    for i in range(len(df_all_ent['Entity'])):
        scores = []
        for j in range(len(df_all_ent['Entity'])):
            scores.append(fuzz.token_sort_ratio(df_all_ent['Entity'][i], df_all_ent['Entity'][j]))

        scores_c = scores.copy()
        scores_c.sort()
        if len(scores_c) > 1:
            sec_max_value = scores_c[-2] 
        else:
            sec_max_value = scores_c[-1]

        max_value = scores_c[-1]

        if sec_max_value >= 75:
            df_all_ent.loc[i, 'index'] = scores.index(sec_max_value)
        else:
            df_all_ent.loc[i, 'index'] = scores.index(max_value)

    # adding similar entities to a column named 'similar'        
    df_all_ent['similar'] = [df_all_ent['Entity'][i] for i in df_all_ent['index']]

    # adding original index
    df_all_ent['index_org'] = df_all_ent.index
    df_all_ent['index'] = df_all_ent['index'].astype(int)


    df_vc = pd.DataFrame({'Indexx':df_all_ent['index'].value_counts().index, "Count":df_all_ent['index'].value_counts()})
    df_vc.reset_index(drop=True, inplace=True)

    # dataframe of index values with count equal to 1 
    df_vcf = df_vc[df_vc['Count']==1]

    # creating column with final index
    df_all_ent['index_fin'] = np.nan
    for i in range(len(df_all_ent['index'])):
        if df_all_ent['index'][i] in df_vcf['Indexx']:
            df_all_ent['index_fin'][i] = min(df_all_ent['index_org'][i], df_all_ent['index'][i])
        else:
            df_all_ent['index_fin'][i] = df_all_ent['index'][i]

    # creating column which contain final entities
    df_all_ent['Entity_final'] = [df_all_ent['Entity'][i] for i in df_all_ent['index_fin']]

    df_final_ent_1 = df_all_ent.copy()
    df_final_ent_1.drop(columns=['index', 'similar', 'index_org', 'index_fin'], inplace=True)

    df_final_ent_2 = df_final_ent_1.copy()
    df_final_ent_2.drop(columns=['Entity'], inplace=True)
      
    df_final_ent = df_final_ent_2.groupby(by=['Entity_final']).agg({'Negative':'sum', 'Neutral':'sum', 'Positive':'sum'})
    df_final_ent.sort_values('Neutral', ascending=False, inplace=True)
    print("\n")
    print("Top 10 dynamic entities along with their sentiment counts: \n")
    print(tabulate(df_final_ent.head(10), headers='keys', tablefmt='fancy_grid'))
    print("\n")
    
    return df

In [ ]:
def save_tweets_of_dynamic_entities(df):
    # a dataframe for tweets information
    df_ti = df[['Entity', 'Sentiment', 'Tweets']]

    df_ti['Entity'] = df_ti['Entity'].str.lower().str.strip().str.replace(" ","")

    # remove entites containing numbers
    for i in range(len(df_ti['Entity'])):
        if any(str(j) in str(df_ti['Entity'][i]) for j in list(range(10))):
            df_ti.loc[i, 'Entity'] = np.nan
        else:
            continue

    df_ti.dropna(inplace=True)
    df_ti.reset_index(inplace=True, drop=True)

    # to find out indices of similar entities
    for i in range(len(df_ti['Entity'])):
        #print(df['Entity'][i])
        scores = []
        for j in range(len(df_ti['Entity'])):
            scores.append(fuzz.token_sort_ratio(df_ti['Entity'][i], df_ti['Entity'][j]))

        scores_c = scores.copy()
        scores_c.sort()
        if len(scores_c) > 1:
            sec_max_value = scores_c[-2] 
        else:
            sec_max_value = scores_c[-1]

        max_value = scores_c[-1]

        if sec_max_value >= 75:
            df_ti.loc[i, 'index'] = scores.index(sec_max_value)
        else:
            df_ti.loc[i, 'index'] = scores.index(max_value)

    # adding similar entities to a column named 'similar'        
    df_ti['similar'] = [df_ti['Entity'][i] for i in df_ti['index']]

    # adding original index
    df_ti['index_org'] = df_ti.index
    df_ti['index'] = df_ti['index'].astype(int)

    # value counts dataframe
    df_vc_ti = pd.DataFrame({'Indexx':df_ti['index'].value_counts().index, "Count":df_ti['index'].value_counts()})
    df_vc_ti.reset_index(drop=True, inplace=True)

    # dataframe of index values with count equal to 1 
    df_vcf_ti = df_vc_ti[df_vc_ti['Count']==1]

    # creating column with final index
    df_ti['index_fin'] = np.nan
    for i in range(len(df_ti['index'])):
        if df_ti['index'][i] in df_vcf_ti['Indexx']:
            df_ti['index_fin'][i] = min(df_ti['index_org'][i], df_ti['index'][i])
        else:
            df_ti['index_fin'][i] = df_ti['index'][i]

    # creating column which contain final entities
    df_ti['Entity_final'] = [df_ti['Entity'][i] for i in df_ti['index_fin']]

    df_ti2 = df_ti.copy()
    df_ti2.drop(columns=['index', 'similar', 'index_org', 'index_fin', 'Entity'], inplace=True)
    import datetime
    df_ti2.set_index(['Entity_final', 'Sentiment']).to_excel('AllTweets_'+str(datetime.datetime.now().replace(microsecond=0)).replace(" ", '_').replace(":", "-")+'.xlsx')


In [ ]:
def print_retweets(df):
    print("Top Retweeted Tweets are: \n")
    j = 1
    retweets_df = df.sort_values("Retweet_count", ascending=False, ignore_index=True)[['Tweets', 'Retweet_count']][:5]
    for index, row in retweets_df.iterrows():
        print(j, end=' ')
        print(row['Tweets']) 
        print("Retweet count: ", row['Retweet_count'])
        print("\n")
        j+=1
    print("\n") 

def print_liked_tweets(df):
    print("Top Liked Tweets are: \n")
    k = 1
    likes_df = df.sort_values("Likes_count", ascending=False, ignore_index=True)[['Tweets', 'Likes_count']][:5]
    for index, row in likes_df.iterrows():
        print(k, end=' ')
        print(row['Tweets']) 
        print("Likes count: ", row['Likes_count'])
        print("\n")
        k+=1
    print("\n")
    
def print_negative_retweets(df):
    k = 1
    negRet_df = df.sort_values("Retweet_count", ascending=False, ignore_index=True)[['Tweets', 'Retweet_count', 'Sentiment']][:5]
    negRet_df2 = negRet_df[negRet_df['Sentiment']=='Negative']
    if len(negRet_df2) > 0:
        print("Top Negative Retweeted tweets are: \n")
        for index, row in negRet2_df.iterrows():
            print(k, end=' ')
            print(row['Tweets']) 
            print("Retweeted count: ", row['Likes_count'])
            print("\n")
            k+=1
    else:
        print("There are no negative retweeted tweets.")
        print("\n")

In [ ]:
# if search word = CSMIA_official
inputText = widgets.Text(description='Search Word: ')

def word_search(keyword):
    with output_box:
        clear_output()
        
        consumer_key = os.environ['TWITTER_CONSUMER_KEY']
        consumer_secret = os.environ['TWITTER_CONSUMER_SECRET']
        access_token = os.environ['TWITTER_ACCESS_TOKEN']
        access_token_secret = os.environ['TWITTER_ACCESS_TOKEN_SECRET']

        search_words = inputText.value + " -filter:retweets"

        # Collect tweets and associated information for entered search word and store it in an object named 'tweets'
        tweets = collect_tweets(consumer_key, consumer_secret, access_token, access_token_secret, search_words)
        
        # Print top 10 recent tweets
        print_ten_tweets(tweets)
        
        # creating dataframe of collected (250) tweets from tweepy
        # containing 4 columns: 'Date', 'Tweets', 'Retweet_count', 'Likes_count'
        df_tweepy = tweets_dataframe(tweets)
        
        # to preprocess the collected tweets
        df_tweepy_processed = preprocess(df_tweepy)
        
        # to print average polarity, subjectivity scores of collected tweets, and
        # to plot quantile counts and compute polarity score percentiles
        ps_scores(df_tweepy_processed)   

        client = Twarc2(bearer_token=os.environ['TWITTER_BEARER_TOKEN'])
        search_words_wo_filter = inputText.value
        
        # to plot daily and hourly count of tweets for last 7 days
        daily_hourly_trend_last7days(client, search_words_wo_filter)           
        
        # to predict sentiment categories of collected and processed tweets
        df_labelled = predict_sentiment_cat(df_tweepy_processed)
        df_labelled = df_labelled.dropna()
         
        file_loc = 'consolidated_till2021-11-30.xlsx'
        dataset = pd.read_excel(file_loc, parse_dates=[2])
        # to get a dataframe of sentiment counts (Oct 1 onwards)
        df_trend = append_tweets_to_consolidated_dataset(dataset, df_labelled)
        
        # to plot horizontal sentiment-wise stacked chart of tweets for last 7 days
        stacked_chart(df_trend)
        
        # to plot word clouds
        plot_wordclouds(df_labelled)
        
        # to plot overall sentiment trend chart
        sentiment_trend(df_trend)
        
        # to preprocess collected tweets for the purpose of entity extraction
        df_processed_ent = preprocess_for_entity(df_labelled)
        
        # to extract entities and thereby print top 10 dynamic 'processed' entities along with their sentiment counts
        df_ent_f = dynamic_entity_table(df_processed_ent)
        
        # to save all the corresponding tweets of dynamic entities along with their sentiment categories in an excel file
        save_tweets_of_dynamic_entities(df_ent_f) 
        
        # to print count of defined entities 
        entities_df = pd.read_excel("pre-defined_entities.xlsx")
        print("Pre-defined entities along with their sentiment counts: \n")
        print(tabulate(final_predefined_entities_count(df_labelled, entities_df), headers='keys', tablefmt='fancy_grid'))
        #print(final_predefined_entities_count(df_labelled, entities_df))
        print("\n")
    
        # to print top 5 retweets
        print_retweets(df_tweepy)
        # to print top 5 liked tweets
        print_liked_tweets(df_tweepy)
        # to print top 5 negative retweets
        print_negative_retweets(df_labelled)        
        

output_box = widgets.Output()
display(output_box)

inputText.on_submit(word_search)

inputText